# 03 Tour of the processed tables

Every table in `data/processed/`, rebuilt by `python src/build_all.py` (automatic after `git pull`).

**How to use this notebook**
1. Run the first two cells, then work down. Each section has a starter cell and **questions to answer yourself**: write a cell, look at the result, note what you learn in DATA.md if it's new.
2. To browse any table visually: **Variables** (notebook toolbar) → Data Wrangler icon next to it. It's best on a filtered slice, e.g. one player or one gameweek, not 60k rows.
3. `.T` flips one row on its side, which is the easiest way to read a single row with 40+ columns.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 60)
PROCESSED = Path.cwd().parent / "data" / "processed"

names = ["player_gw", "player_match", "fixtures", "players", "teams", "players_daily", "gameweeks"]
tables = {n: pd.read_parquet(PROCESSED / f"{n}.parquet") for n in names}
player_gw, player_match, fixtures, players, teams, players_daily, gameweeks = tables.values()

for n, df in tables.items():
    print(f"{n:14} {df.shape[0]:>7,} rows x {df.shape[1]:>3} cols")

player_gw       60,909 rows x  42 cols
player_match    60,923 rows x  29 cols
fixtures         1,140 rows x  13 cols
players          2,292 rows x   8 cols
teams               60 rows x   5 cols
players_daily    6,670 rows x  13 cols
gameweeks          114 rows x   4 cols


## Names on any table

Tables store stable codes, not names: each fact once (see DATA.md). This helper joins names when *you* need to read results.

In [2]:
def with_names(df):
    """Add web_name (and team_name if there's a team_code) by joining the lookups."""
    out = df.merge(players[["season", "player_code", "web_name"]], on=["season", "player_code"], how="left")
    if "team_code" in out:
        out = out.merge(teams[["season", "team_code", "team_name"]], on=["season", "team_code"], how="left")
    return out

def find(name):
    """player_code(s) for a name, e.g. find("Haaland")."""
    return players[players["web_name"].str.contains(name, case=False)][["season", "player_code", "web_name", "position"]]

find("Haaland")

,season,player_code,web_name,position
481,2024-25,223094,Haaland,FWD
1309,2025-26,223094,Haaland,FWD
2116,2026-27,223094,Haaland,FWD


## 1. `player_gw`: the main modelling table (one row per player per gameweek)

Column groups: **keys** (season, player_code, gw) · **targets** (what happened: minutes, goals, total_points …) · **`f_*`** (form from *earlier* gameweeks) · **`snap_*`** (latest snapshot *before* the deadline) · **context** (team, position, n_fixtures, n_home, deadline) · **`fpl_xp`** (FPL's own prediction, past seasons).

In [ ]:
code = 223094   # Haaland; change it using find(...)
cols = ["season", "gw", "n_fixtures", "minutes", "total_points", "f_minutes_mean_3", "f_xg_per90_5",
        "snap_status", "snap_ep_next", "fpl_xp"]
player_gw[player_gw["player_code"] == code][cols].tail(12)

**Questions**
1. Find one **blank** gameweek row and one **double** (`n_fixtures` 0 and 2). What do the targets look like in each, and why?
2. Pick any player's **first-ever** row. Which columns are NaN, and why is that correct (think leakage)?
3. For one row, compare `f_minutes_mean_3` with the minutes of the three gameweeks *before* it. Do they match? What about the current gameweek's minutes?
4. Which columns are **targets** and which are **features**? If you trained a model to predict `total_points`, which columns must you never feed it?

## 2. `players_daily`: what was known each day (one row per player per snapshot)

In [ ]:
snap = players_daily[players_daily["player_code"] == code]
snap[["snapshot_time", "status", "chance_of_playing_next_round", "now_cost", "selected_by_percent", "ep_next", "news"]]

**Questions**
1. Find players whose `status` **changed** between the first and the latest snapshot (e.g. `a` → `i`). Who got injured this week? *Hint:* `groupby("player_code")["status"].agg(["first", "last"])`.
2. Who gained the most ownership (`selected_by_percent`) since the first snapshot?
3. Look at `ep_next` around a deadline (`gameweeks` has them). Why does it jump right after the deadline, and why does that make the "strictly before" as-of join necessary?

## 3. `fixtures`: every match, played and future

In [ ]:
upcoming = fixtures[(fixtures["season"] == "2026-27") & ~fixtures["finished"]]
upcoming[upcoming["gw"] <= upcoming["gw"].min() + 5][
    ["gw", "kickoff_time", "team_h_name", "team_a_name", "team_h_difficulty", "team_a_difficulty"]].head(12)

**Questions**
1. Over the next 6 gameweeks, how many fixtures does each team have? Any **doubles or blanks** yet? *Hint:* stack home and away teams into one column, then `groupby(["gw", "team"]).size()`.
2. Which team has the **easiest** next 6 by FPL difficulty? Do you trust FPL's difficulty numbers? (Phase 1 builds its own.)
3. For a finished 2025-26 match, check that `player_match` points for both teams' players make sense with the score (e.g. clean sheets for the side that conceded 0).

## 4. `player_match` → a first look at team strength

Team strength (Phase 1) works on **one row per team per match**. You can build it from `player_match` by summing players.

In [ ]:
cur = player_match[player_match["season"] == "2026-27"]
team_xg = cur.groupby(["fixture", "team_code"], as_index=False)["expected_goals"].sum()
team_xg.head()

**Questions**
1. Add **xG against** to `team_xg` (the *opponent's* xG in the same fixture). *Hint:* merge `team_xg` with itself on `fixture`, keeping rows where the team codes differ.
2. Rank teams by average xG for and against this season (add names with `teams`). Does the table look like the real league table?
3. Why might summing *player* xG slightly undercount a team's xG? (Think: who isn't in FPL's data.)

## 5. Baselines: how good is FPL's own prediction?

In [ ]:
past = player_gw[player_gw["fpl_xp"].notna() & (player_gw["n_fixtures"] > 0)]
past[["fpl_xp", "total_points"]].describe()

**Questions**
1. Reproduce the bar from DECISIONS.md: MAE of `fpl_xp` vs `total_points` (should be ≈ 0.855 on the same rows; where do your rows differ?).
2. Bucket `fpl_xp` (e.g. `pd.cut`) and compare the average `total_points` per bucket. Is FPL **calibrated**: when it says 6, do players average 6?
3. Is FPL better for some positions than others?